# Day 8 — Solution: Autocorrelation & Newey–West

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - X.shape[1])
    se = np.sqrt(np.diag(s2 * XtX_inv))
    return {"alpha": b[0], "beta": b[1], "se_beta": se[1], "resid": e, "X": X}

def se_nw(X, e, L):
    xe = X * e[:, None]
    meat = xe.T @ xe
    for l in range(1, L + 1):
        w = 1 - l / (L + 1)
        G = xe[l:].T @ xe[:-l]
        meat += w * (G + G.T)
    XtX_inv = np.linalg.inv(X.T @ X)
    return np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv))

## E1 — the ramp, predicted and measured

In [ ]:
rng = np.random.default_rng(41)
r = pd.Series(rng.normal(0, 0.01, 3000))
y = r.rolling(21).sum().shift(-21).dropna()
# prediction: ACF(l) = (21 - l) / 21 for l <= 21, else 0
pred = {l: max((21 - l) / 21, 0) for l in [1, 10, 20, 21, 40]}
meas = {l: y.autocorr(l) for l in [1, 10, 20, 21, 40]}
print(pd.DataFrame([pred, meas], index=["predicted", "measured"]).T.round(3).to_string())

**Expected pattern.** Measured tracks predicted within ~±0.03 at every lag
— the overlap's ACF is *pure arithmetic*, the shared-days fraction, and it
dies exactly at lag 21. The diagnostic rule for the colleague: "an ACF
ramping linearly to zero at lag q means a q-period window is re-using the
data; set L ≥ q and E2's tournament is what happens if you don't."
Notably: correlation this large (0.95 at lag 1!) from a process with ZERO
predictive content — autocorrelation is about measurement architecture,
not signal (module 08 will chase the subtler versions).

## E2 — the tournament of lags

In [ ]:
n_runs, n = 600, 3000
lags = [0, 5, 21, 42]
est = np.empty(n_runs)
ses = {l: np.empty(n_runs) for l in lags}
for i in range(n_runs):
    rng = np.random.default_rng(12000 + i)
    r = rng.normal(0, 0.01, n)
    x = pd.Series(r).rolling(21).sum().shift(1)          # trailing 21d (persistent)
    y = pd.Series(r).rolling(21).sum().shift(-21)        # forward 21d (fresh noise beyond x)
    d = pd.concat([x, y], axis=1, keys=["x", "y"]).dropna()
    f = ols_fit(d["x"].values, d["y"].values)
    est[i] = f["beta"]
    for l in lags:
        ses[l][i] = f["se_beta"] if l == 0 else se_nw(f["X"], f["resid"], l)[1]

sd_true = est.std(ddof=1)
print(f"true sd(beta_hat) = {sd_true:.4f}   (E[beta] {est.mean():+.4f} vs truth 0)")
for l in lags:
    rej = (np.abs(est) / ses[l] > 1.96).mean()
    print(f"Lag {l:2d}: mean SE {ses[l].mean():.4f}  |  rejection {rej:.3f}")

**Expected pattern.** β̂ unbiased throughout (mean ≈ 0 ± 2/√600 SEs) —
autocorrelation never touches the point estimate under A2. The SE column
tells the story: White (L = 0) mean SE roughly 2–3× too small → rejection
60–65%; **L = 5 still rejects ~28%** (it repaired a fraction of a 21-lag
mechanism); L = 21 ≈ L = 42 → rejection ≈ 0.10–0.14 with mean SE now
tracking the true sd — better, but not the 5% you paid for. That residue
is a FACT about HAC, not your bug: consistent, not exact — at
near-unit-root autocorrelation the sandwich under-covers at this n, and
raising L stays only mildly ahead (L = 63: ~0.10). The receipts are blunt:
*the lag parameter is not a detail* — an under-lagged NW gives false
comfort at research scale — and for THIS construction the exact repair
isn't a longer sandwich, it's E4's non-overlapped re-estimate, which NW
is doing its asymptotic best to mimic.

## E3 — hand vs statsmodels

In [ ]:
rng = np.random.default_rng(12000)
r = rng.normal(0, 0.01, 3000)
x = pd.Series(r).rolling(21).sum().shift(1)
y = pd.Series(r).rolling(21).sum().shift(-21)
d = pd.concat([x, y], axis=1, keys=["x", "y"]).dropna()
f = ols_fit(d["x"].values, d["y"].values)

import statsmodels.api as sm
sm_hac = sm.OLS(d["y"].values, sm.add_constant(d["x"].values)).fit(
    cov_type="HAC", cov_kwds={"maxlags": 21})
mine = se_nw(f["X"], f["resid"], 21)[1]
sm_bse = np.asarray(sm_hac.bse)
print(f"hand NW {mine:.5f}  |  statsmodels HAC {sm_bse[1]:.5f}  |  "
      f"diff {abs(mine - sm_bse[1]) / sm_bse[1]:.2%}")

**Expected pattern.** Agreement to ≲1% (any tiny gap is a degrees-of-
freedom flag, not a formula difference). Third library demystified: White
was your meat diagonal, HAC is your meat with memory. **You now own the
entire classical→White→HAC ladder — bread constant, meat matched to the
violation.**

## E4 — the long-horizon regression, two ways

In [ ]:
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    p = get_prices("SPY", start="1993-01-01")["SPY"]
else:
    p = synthetic_prices(n_days=8000, n_assets=1, seed=4)["S0"]
lp = np.log(p)

fwd = lp.shift(-21) - lp
trail = lp - lp.shift(63)
d = pd.concat([trail, fwd], axis=1, keys=["trail63", "fwd21"]).dropna()
g = ols_fit(d["trail63"].values, d["fwd21"].values)
t_class = g["beta"] / g["se_beta"]
t_nw = g["beta"] / se_nw(g["X"], g["resid"], 21)[1]
print(f"overlapped monthly: beta {g['beta']:+.3f}   t(classical) {t_class:+.2f}   t(NW, L=21) {t_nw:+.2f}")

pm = p.resample("ME").last()
rm = np.log(pm).diff()
dm = pd.concat([rm.rolling(3).sum().shift(1), rm], axis=1, keys=["trail3m", "m"]).dropna()
g2 = ols_fit(dm["trail3m"].values, dm["m"].values)
print(f"non-overlapped monthly: beta {g2['beta']:+.3f}   t(classical, honest here) {g2['beta']/g2['se_beta']:+.2f}   n {len(dm)}")

**Expected pattern.** In BOTH modes the predictor is near-worthless
(real SPY: index-level monthly momentum is at most weak; synthetic: no
predictability planted), but the overlapped classical |t| lands visibly
higher than NW's — same data, same β̂, different honesty. The
non-overlapped version: t ≈ NW's answer with ~n/21 observations. **The
overlap bought apparent sample size, not information — twenty-one
observations that are 95% shared are one observation wearing 21
costumes; the repair is pricing exactly that.** Common mistake: reporting
the overlapped table because the non-overlapped one "loses data" — the
data was never there; there were never 3000 independent monthly returns
in 3000 days.

## E5 — reading the famous tables (exemplar)

Discipline: with 3-month overlap, the mechanical inflation ceiling on a
classical t is ~√3 ≈ 1.7× — but momentum papers' portfolios add
*cross-sectional* overlap too (the same stock appears in overlapping
portfolios), so a mental 2–3× is the safe haircut absent a labeled SE.
Plausible flavors in a careful paper: NW/HAC at L ≈ 3–12 on monthly data,
or Fama–MacBeth SEs (day 15 — the portfolio time series IS the regression
input there, and FM is the field-standard repair). t = 9 survives every
haircut: even 9/3 = 3 — the finding is robust to SE-flavor ignorance,
which is exactly how you want a flagship result to behave. t = 2.4 in the
same table: unlabeled and unverifiable — treat as *negative until proven
positive*: ask for the SE flavor, and apply the L-rule mentally (2.4
classical at 3-month overlap plausibly ≈ 1.4 honest). **The reading skill
is not suspicion for its own sake — it is pricing the table's t-stats in
the currency the data structure demands.**